# Week 2 — GradCAM + LIME (Person 1)
Run cells 1→2→3→4→5→6→7→8 in order. Do not skip.

In [ ]:
# Cell 1: Mount Drive + Clone Repo
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, sys
REPO_URL = 'https://github.com/rao-shreyashree/diabetic-retinopathy-xai.git'
REPO_DIR = '/content/diabetic-retinopathy-xai'
if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull')
sys.path.insert(0, REPO_DIR)
print('Repo ready')

In [ ]:
# Cell 2: Install Dependencies
import os
os.system('pip install timm lime scikit-image -q')
print('Done')

In [ ]:
# Cell 3: Imports + Config
import os, json, csv
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
from utils.config import (CKPT, HEATMAP_DIR, PREDICTIONS_CSV,
                           GRADCAM_LAYER, GRADING_TEST_DIR)
from datasets.idrid_dataset import IDRiDTestDataset
from xai.gradcam import GradCAM, build_model_for_xai
from xai.lime_wrapper import LIMEWrapper
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
with open(os.path.join(REPO_DIR, 'test_image_ids.json')) as f:
    TEST_IDS = json.load(f)
print(f'{len(TEST_IDS)} test images loaded')

In [ ]:
# Cell 4: Create Output Directories
MODELS  = ['efficientnetb4', 'resnet50']
METHODS = ['gradcam', 'lime']
for model_name in MODELS:
    for method in METHODS:
        os.makedirs(os.path.join(HEATMAP_DIR, model_name, method), exist_ok=True)
os.makedirs(os.path.dirname(PREDICTIONS_CSV), exist_ok=True)
print('Output dirs ready')

In [ ]:
# Cell 5: Load Dataset
dataset = IDRiDTestDataset()
print(f'Loaded {len(dataset)} test images')
tensor, label, image_id, pil_img = dataset[0]
print(f'Sample: {image_id}, label={label}, tensor={tensor.shape}, PIL={pil_img.size}')

In [ ]:
# Cell 6: Generate GradCAM + LIME for all images x both models
all_prediction_rows = []

for model_name in MODELS:
    print(f'\n=== {model_name.upper()} ===')
    model = build_model_for_xai(model_name, CKPT[model_name], DEVICE)

    # GradCAM
    print('-- GradCAM --')
    cam         = GradCAM(model, target_layer_name=GRADCAM_LAYER[model_name], device=DEVICE)
    gradcam_dir = os.path.join(HEATMAP_DIR, model_name, 'gradcam')
    for i in range(len(dataset)):
        tensor, true_label, image_id, pil_img = dataset[i]
        try:
            heatmap, pred_class = cam.generate(tensor, pil_img)
            assert heatmap.shape == (pil_img.size[1], pil_img.size[0])
            assert heatmap.dtype == np.float32
            np.save(os.path.join(gradcam_dir, f'{model_name}_gradcam_{image_id}.npy'), heatmap)
            with torch.no_grad():
                logits = model(tensor.unsqueeze(0).to(DEVICE))
                conf   = torch.softmax(logits, dim=1)[0][pred_class].item()
            all_prediction_rows.append({'image_id': image_id, 'model': model_name,
                'predicted_grade': pred_class, 'true_grade': int(true_label),
                'confidence': round(conf, 4)})
            print(f'  [{i+1:02d}/27] {image_id} pred={pred_class} true={true_label}')
        except Exception as e:
            print(f'  [{i+1:02d}/27] {image_id} FAILED: {e} — skipping')

    # LIME
    print('-- LIME (slow ~2-3 min/image) --')
    lime_exp  = LIMEWrapper(model, device=DEVICE)
    lime_dir  = os.path.join(HEATMAP_DIR, model_name, 'lime')
    for i in range(len(dataset)):
        tensor, true_label, image_id, pil_img = dataset[i]
        try:
            heatmap, pred_class = lime_exp.generate(pil_img)
            assert heatmap.shape == (pil_img.size[1], pil_img.size[0])
            assert heatmap.dtype == np.float32
            np.save(os.path.join(lime_dir, f'{model_name}_lime_{image_id}.npy'), heatmap)
            print(f'  [{i+1:02d}/27] {image_id} pred={pred_class}')
        except Exception as e:
            print(f'  [{i+1:02d}/27] {image_id} FAILED: {e} — skipping')

    del model
    torch.cuda.empty_cache()

# Write predictions CSV
with open(PREDICTIONS_CSV, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['image_id','model','predicted_grade','true_grade','confidence'])
    writer.writeheader()
    writer.writerows(all_prediction_rows)
print(f'predictions.csv written — {len(all_prediction_rows)} rows')

In [ ]:
# Cell 7: Visual Sanity Check — 3 sample images
import random
random.seed(42)
samples = random.sample(TEST_IDS, 3)

def overlay(pil_img, heatmap, alpha=0.5):
    colored = (cm.jet(heatmap)[:,:,:3] * 255).astype(np.uint8)
    orig    = np.array(pil_img.resize((heatmap.shape[1], heatmap.shape[0])))
    return Image.fromarray((alpha*orig + (1-alpha)*colored).astype(np.uint8))

for image_id in samples:
    pil = Image.open(os.path.join(GRADING_TEST_DIR, image_id+'.jpg')).convert('RGB')
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(image_id, fontsize=14)
    for row, model_name in enumerate(MODELS):
        axes[row][0].imshow(pil); axes[row][0].set_title(f'{model_name}\nOriginal'); axes[row][0].axis('off')
        for col, method in enumerate(['gradcam','lime'], 1):
            p = os.path.join(HEATMAP_DIR, model_name, method, f'{model_name}_{method}_{image_id}.npy')
            if os.path.exists(p):
                axes[row][col].imshow(overlay(pil, np.load(p)))
                axes[row][col].set_title(method.upper())
            else:
                axes[row][col].text(0.5, 0.5, 'MISSING', ha='center', va='center')
            axes[row][col].axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
# Cell 8: File Count Check — verify everything saved correctly
import glob
expected = 27
all_ok   = True
print('File counts:')
for model_name in MODELS:
    for method in METHODS:
        files  = glob.glob(os.path.join(HEATMAP_DIR, model_name, method, '*.npy'))
        status = 'OK' if len(files)==expected else f'MISSING {expected-len(files)}'
        print(f'  {model_name}/{method}: {len(files)}/{expected} [{status}]')
        if len(files) != expected: all_ok = False
pred_rows = sum(1 for _ in open(PREDICTIONS_CSV)) - 1
print(f'  predictions.csv: {pred_rows} rows (expected {expected*len(MODELS)})')
print('\nP1 COMPLETE - push to GitHub!' if all_ok else '\nSome files missing - check Cell 6 for errors.')